In [ ]:
!pip install gradio

In [ ]:
import sqlite3

# Create library database
conn = sqlite3.connect("library.db")
cursor = conn.cursor()

# Create books table
cursor.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    category TEXT,
    quantity INTEGER NOT NULL,
    available INTEGER NOT NULL
)
""")

# Create students table
cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    department TEXT,
    phone TEXT
)
""")

# Create transactions table
cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    book_id INTEGER,
    student_id INTEGER,
    issue_date TEXT,
    due_date TEXT,
    return_date TEXT,
    fine REAL DEFAULT 0
)
""")

conn.commit()
conn.close()

print("Library database created successfully!")
print("Books, Students and Transactions tables are ready.")

Library database created successfully!
Books, Students and Transactions tables are ready.


In [ ]:
import sqlite3

def add_book(title, author, category, quantity):
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO books
        (title, author, category, quantity, available)
        VALUES (?, ?, ?, ?, ?)
    """, (title, author, category, quantity, quantity))

    conn.commit()
    conn.close()

    return "Book added successfully!"


def view_books():
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM books")
    records = cursor.fetchall()

    conn.close()

    return records


def search_book(keyword):
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT * FROM books
        WHERE title LIKE ? OR author LIKE ?
    """, ('%' + keyword + '%', '%' + keyword + '%'))

    records = cursor.fetchall()
    conn.close()

    return records


print("Book management functions created successfully!")

Book management functions created successfully!


In [ ]:
import sqlite3
from datetime import date, timedelta


# STUDENT REGISTRATION

def register_student(name, department, phone):
    if not name.strip():
        return "Please enter student name."

    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO students (name, department, phone)
        VALUES (?, ?, ?)
    """, (name, department, phone))

    conn.commit()
    conn.close()

    return "Student registered successfully!"


# VIEW STUDENTS

def view_students():
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM students")
    records = cursor.fetchall()

    conn.close()
    return records


# ISSUE BOOK

def issue_book(book_id, student_id):
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    book = cursor.execute("""
        SELECT available FROM books WHERE book_id=?
    """, (book_id,)).fetchone()

    student = cursor.execute("""
        SELECT student_id FROM students WHERE student_id=?
    """, (student_id,)).fetchone()

    if not book:
        conn.close()
        return "Book ID not found."

    if not student:
        conn.close()
        return "Student ID not found."

    if book[0] <= 0:
        conn.close()
        return "Book is not available."

    issue_date = date.today()
    due_date = issue_date + timedelta(days=14)

    cursor.execute("""
        INSERT INTO transactions
        (book_id, student_id, issue_date, due_date)
        VALUES (?, ?, ?, ?)
    """, (
        book_id,
        student_id,
        issue_date.isoformat(),
        due_date.isoformat()
    ))

    cursor.execute("""
        UPDATE books
        SET available = available - 1
        WHERE book_id=?
    """, (book_id,))

    conn.commit()
    conn.close()

    return f"Book issued successfully! Due date: {due_date}"


# RETURN BOOK

def return_book(transaction_id):
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    transaction = cursor.execute("""
        SELECT book_id, due_date
        FROM transactions
        WHERE transaction_id=? AND return_date IS NULL
    """, (transaction_id,)).fetchone()

    if not transaction:
        conn.close()
        return "Active transaction not found."

    book_id, due_date = transaction

    return_date = date.today()
    due = date.fromisoformat(due_date)

    overdue_days = max(0, (return_date - due).days)
    fine = overdue_days * 2

    cursor.execute("""
        UPDATE transactions
        SET return_date=?, fine=?
        WHERE transaction_id=?
    """, (return_date.isoformat(), fine, transaction_id))

    cursor.execute("""
        UPDATE books
        SET available=available+1
        WHERE book_id=?
    """, (book_id,))

    conn.commit()
    conn.close()

    return f"Book returned successfully! Fine: Rs. {fine}"


print("Student registration completed!")
print("Book issue and return functions created successfully!")

Student registration completed!
Book issue and return functions created successfully!


In [ ]:
import gradio as gr
import sqlite3


# DISPLAY BOOKS

def display_books():
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT book_id, title, author, category,
               quantity, available
        FROM books
        ORDER BY book_id
    """)

    records = cursor.fetchall()
    conn.close()
    return records


# DISPLAY STUDENTS

def display_students():
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT student_id, name, department, phone
        FROM students
        ORDER BY student_id
    """)

    records = cursor.fetchall()
    conn.close()
    return records


# DISPLAY TRANSACTIONS

def display_transactions():
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT
            t.transaction_id,
            b.title,
            s.name,
            t.issue_date,
            t.due_date,
            t.return_date,
            t.fine
        FROM transactions t
        JOIN books b ON t.book_id = b.book_id
        JOIN students s ON t.student_id = s.student_id
        ORDER BY t.transaction_id DESC
    """)

    records = cursor.fetchall()
    conn.close()
    return records


# DASHBOARD STATISTICS

def dashboard_stats():
    conn = sqlite3.connect("library.db")
    cursor = conn.cursor()

    cursor.execute("SELECT COALESCE(SUM(quantity), 0) FROM books")
    total_books = cursor.fetchone()[0]

    cursor.execute("SELECT COALESCE(SUM(available), 0) FROM books")
    available = cursor.fetchone()[0]

    cursor.execute("SELECT COUNT(*) FROM students")
    students = cursor.fetchone()[0]

    cursor.execute("""
        SELECT COUNT(*) FROM transactions
        WHERE return_date IS NULL
    """)
    issued = cursor.fetchone()[0]

    cursor.execute("SELECT COALESCE(SUM(fine), 0) FROM transactions")
    fines = cursor.fetchone()[0]

    conn.close()

    return (
        f"### Total book copies: {total_books}\n"
        f"### Available copies: {available}\n"
        f"### Registered students: {students}\n"
        f"### Currently issued: {issued}\n"
        f"### Fines recorded: Rs. {fines:.2f}"
    )


# SEARCH BOOKS

def search_books_ui(keyword):
    if not keyword:
        return display_books()

    return search_book(keyword)


# CREATE WEB APPLICATION

with gr.Blocks(title="Library Management System") as app:

    gr.Markdown("""
    # 📚 LIBRARY MANAGEMENT SYSTEM

    ### Manage books, students, book issuing and returns
    """)

    with gr.Tab("Dashboard"):
        dashboard_output = gr.Markdown(
            dashboard_stats()
        )

        refresh_dashboard = gr.Button("Refresh Dashboard")
        refresh_dashboard.click(
            dashboard_stats,
            outputs=dashboard_output
        )

    with gr.Tab("Book Management"):

        gr.Markdown("## Add New Book")

        title_input = gr.Textbox(label="Book Title")
        author_input = gr.Textbox(label="Author")
        category_input = gr.Textbox(label="Category")
        quantity_input = gr.Number(
            label="Quantity",
            precision=0,
            value=1
        )

        add_button = gr.Button("Add Book", variant="primary")
        add_message = gr.Textbox(label="Status")

        def add_book_ui(title, author, category, quantity):
            if not title or not title.strip():
                return "Please enter a book title."

            if not author or not author.strip():
                return "Please enter an author."

            if quantity is None or quantity < 0 or int(quantity) != quantity:
                return "Enter a valid non-negative whole quantity."

            return add_book(
                title.strip(),
                author.strip(),
                category.strip() if category else "",
                int(quantity)
            )

        add_button.click(
            add_book_ui,
            inputs=[
                title_input,
                author_input,
                category_input,
                quantity_input
            ],
            outputs=add_message
        )

        gr.Markdown("## Search Books")

        search_input = gr.Textbox(label="Search by title or author")
        search_button = gr.Button("Search")

        book_table = gr.Dataframe(
            headers=[
                "Book ID", "Title", "Author",
                "Category", "Total", "Available"
            ],
            label="Book Records",
            interactive=False
        )

        search_button.click(
            search_books_ui,
            inputs=search_input,
            outputs=book_table
        )

        show_books_button = gr.Button("Show All Books")

        show_books_button.click(
            display_books,
            outputs=book_table
        )

    with gr.Tab("Student Management"):

        gr.Markdown("## Register Student")

        student_name = gr.Textbox(label="Student Name")
        department = gr.Textbox(label="Department")
        phone = gr.Textbox(label="Phone Number")

        register_button = gr.Button(
            "Register Student",
            variant="primary"
        )

        student_message = gr.Textbox(label="Status")

        def register_student_ui(name, dept, mobile):
            if not name or not name.strip():
                return "Please enter student name."

            return register_student(
                name.strip(),
                dept.strip() if dept else "",
                mobile.strip() if mobile else ""
            )

        register_button.click(
            register_student_ui,
            inputs=[student_name, department, phone],
            outputs=student_message
        )

        gr.Markdown("## Student Records")

        student_table = gr.Dataframe(
            headers=[
                "Student ID", "Name", "Department", "Phone"
            ],
            interactive=False
        )

        show_students_button = gr.Button("Show Students")

        show_students_button.click(
            display_students,
            outputs=student_table
        )

    with gr.Tab("Issue Book"):

        gr.Markdown("## Issue a Book")

        issue_book_id = gr.Number(
            label="Book ID",
            precision=0
        )

        issue_student_id = gr.Number(
            label="Student ID",
            precision=0
        )

        issue_button = gr.Button(
            "Issue Book",
            variant="primary"
        )

        issue_message = gr.Textbox(label="Issue Status")

        def issue_book_ui(book_id, student_id):
            if book_id is None or student_id is None:
                return "Enter both Book ID and Student ID."

            if book_id <= 0 or student_id <= 0:
                return "IDs must be positive numbers."

            if int(book_id) != book_id or int(student_id) != student_id:
                return "IDs must be whole numbers."

            return issue_book(int(book_id), int(student_id))

        issue_button.click(
            issue_book_ui,
            inputs=[issue_book_id, issue_student_id],
            outputs=issue_message
        )

    with gr.Tab("Return Book"):

        gr.Markdown("## Return an Issued Book")

        return_transaction_id = gr.Number(
            label="Transaction ID",
            precision=0
        )

        return_button = gr.Button(
            "Return Book",
            variant="primary"
        )

        return_message = gr.Textbox(label="Return Status")

        def return_book_ui(transaction_id):
            if transaction_id is None or transaction_id <= 0:
                return "Enter a valid transaction ID."

            if int(transaction_id) != transaction_id:
                return "Transaction ID must be a whole number."

            return return_book(int(transaction_id))

        return_button.click(
            return_book_ui,
            inputs=return_transaction_id,
            outputs=return_message
        )

    with gr.Tab("Transaction History"):

        gr.Markdown("## Book Issue and Return Records")

        transaction_table = gr.Dataframe(
            headers=[
                "Transaction ID",
                "Book",
                "Student",
                "Issue Date",
                "Due Date",
                "Return Date",
                "Fine"
            ],
            interactive=False
        )

        history_button = gr.Button("View Transaction History")

        history_button.click(
            display_transactions,
            outputs=transaction_table
        )


print("Library Management System is ready!")

Library Management System is ready!


In [ ]:
app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d574fa99b77c9539a6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
